# U-Net: heatmap лазерных отметок

Выберите профиль, проверьте целевые карты и запустите обучение. Для Windows используйте `num_workers: 0` в Jupyter. CLI поддерживает многопроцессную загрузку. До фиксации параметров используйте validation. Для финальной оценки установите `PARTITION = "test"`.

In [ ]:
import pandas as pd
from dataclasses import replace
from IPython.display import Image, display

from laser_detection.benchmarking import read_image
from laser_detection.config import load_config
from laser_detection.data import load_dataset
from laser_detection.unet.data import HeatmapDataset, image_tensor, training_split
from laser_detection.unet.inference import Predictor, compare_checkpoint
from laser_detection.unet.plots import figure_png, heatmap_figure
from laser_detection.unet.training import train_run

PROFILE = "cpu"  # "cuda" для подготовленного CUDA-окружения
PARTITION = "validation"
config = load_config()
profile = replace(getattr(config.unet, PROFILE), num_workers=0)
frame = load_dataset(config)
splits = training_split(frame, config, profile.seed)
print(splits["protocol"].iloc[0])
display(splits.groupby("split").size())

## Пример целевой карты с обучающего кадра

In [ ]:
train_rows = frame.loc[frame["image"].isin(splits.loc[splits["split"].eq("train"), "image"])]
image, target, meta = HeatmapDataset(train_rows, config, profile)[0]
display(meta, Image(data=figure_png(heatmap_figure(image.permute(1, 2, 0), target[0]))))

## Обучение и история loss

In [ ]:
run = train_run(frame, config, profile)
history = pd.read_csv(run / "history.csv")
axis = history.plot(x="epoch", y=["train_loss", "validation_loss"], ylabel="Loss", marker="o")
display(axis.figure)
axis.figure.savefig(run / "training_loss.png")
print(run)

## Лучший checkpoint и сравнение с HSV Legacy

In [ ]:
checkpoint = run / "best.pt"
predictor = Predictor(checkpoint, profile.device)
evaluation, tables = compare_checkpoint(frame, config, predictor, partition=PARTITION)
display(tables["metrics"], tables["metrics_by_setup"], tables["timings"])
print(evaluation)

## Тепловая карта и найденные центры

In [ ]:
successful = tables["predictions"].query("method == 'unet' and status == 'ok'")
filename = successful.iloc[0]["image"]  # Можно указать имя другого локального кадра.
bgr = read_image(filename, config)
coordinates, confidences, probability, elapsed = predictor.predict(bgr)
rgb = image_tensor(bgr, profile.image_size).permute(1, 2, 0)
figure = heatmap_figure(rgb, probability, coordinates=coordinates, original_size=bgr.shape[1::-1])
display(filename, coordinates, confidences, Image(data=figure_png(figure)))
figure.savefig(evaluation / "heatmap_example.png")